In [1]:
import pandas as pd

# load the CSVs

form_path  = "./data/pred_form_energy.csv"
gap_path   = "./data/pred_band_gap.csv"
gibbs_path = "./data/pred_gibbs_free.csv"

df_form  = pd.read_csv(form_path)
df_gap   = pd.read_csv(gap_path)
df_gibbs = pd.read_csv(gibbs_path)


In [2]:
# merge into one table

df = (
    df_form.merge(df_gap[["mxene", "predicted_label"]], on="mxene", how="inner")
           .merge(df_gibbs, on="mxene", how="inner")
)

# Clean + derived columns
df["predicted_label"] = df["predicted_label"].astype(str).str.strip().str.lower()
df["abs_predicted_gibbs_free_energy"] = df["predicted_gibbs_free_energy"].abs()


In [3]:
# define thresholds

EFORM_CUTOFF = -0.5   # eV/atom
DG_CUTOFF    =  0.10  # eV  (use abs value)
KEEP_LABEL   = "conductor"

# Filtering masks
m_stable   = df["predicted_formation_energy_per_atom"] <= EFORM_CUTOFF
m_cond     = df["predicted_label"] == KEEP_LABEL
m_active   = df["abs_predicted_gibbs_free_energy"] <= DG_CUTOFF

df["pass_eform"]  = m_stable
df["pass_cond"]   = m_cond
df["pass_dgH"]    = m_active
df["pass_all"]    = m_stable & m_cond & m_active


In [4]:
# create the filtered shortlist + ranking

shortlist = df[df["pass_all"]].copy()

# Rank: 1) smaller |ΔG_H*| is better, 2) more negative formation energy is better
shortlist = shortlist.sort_values(
    by=["abs_predicted_gibbs_free_energy", "predicted_formation_energy_per_atom"],
    ascending=[True, True]
).reset_index(drop=True)

# Optional: tier labels for convenience
shortlist["dg_tier"] = pd.cut(
    shortlist["abs_predicted_gibbs_free_energy"],
    bins=[-1e9, 0.10, 0.20, 0.30],
    labels=["elite (<=0.10)", "high (0.10–0.20)", "good (0.20–0.30)"]
)


In [5]:
# quick reporting

print("Total candidates:", len(df))
print("Pass formation energy (<= {:.2f}): {}".format(EFORM_CUTOFF, int(m_stable.sum())))
print("Pass conductor label:            {}".format(int(m_cond.sum())))
print("Pass |ΔG_H*| (<= {:.2f}):        {}".format(DG_CUTOFF, int(m_active.sum())))
print("Pass ALL (final shortlist):      {}".format(len(shortlist)))

print("\nTop 20 candidates (ranked):")
display_cols = [
    "mxene",
    "predicted_formation_energy_per_atom",
    "predicted_label",
    "predicted_gibbs_free_energy",
    "abs_predicted_gibbs_free_energy",
    "dg_tier",
]
display(shortlist[display_cols].head(20))


Total candidates: 792
Pass formation energy (<= -0.50): 611
Pass conductor label:            696
Pass |ΔG_H*| (<= 0.10):        76
Pass ALL (final shortlist):      48

Top 20 candidates (ranked):


,mxene,predicted_formation_energy_per_atom,predicted_label,predicted_gibbs_free_energy,abs_predicted_gibbs_free_energy,dg_tier
0,Sc4N3N2H2,-1.537453,conductor,-0.004481,0.004481,elite (<=0.10)
1,Zr3N2N2H2,-1.309463,conductor,-0.004729,0.004729,elite (<=0.10)
2,Y3C2O2,-1.952476,conductor,0.005450,0.005450,elite (<=0.10)
3,Cr3N2N2H2,-0.574421,conductor,0.006880,0.006880,elite (<=0.10)
4,Nb3C2N2H2,-0.753476,conductor,-0.008352,0.008352,elite (<=0.10)
5,V2NN2H2,-0.822598,conductor,0.008392,0.008392,elite (<=0.10)
6,Y2CH2,-0.503158,conductor,-0.012999,0.012999,elite (<=0.10)
7,Sc2CN2H2,-1.221955,conductor,0.014039,0.014039,elite (<=0.10)
8,Nb3N2N2H2,-0.756980,conductor,0.014879,0.014879,elite (<=0.10)
9,Ti3N2N2H2,-1.219964,conductor,0.015938,0.015938,elite (<=0.10)


In [6]:
# save outputs

df.to_csv("./data/mxene_flags.csv", index=False)
shortlist.to_csv("./data/mxene_filtered.csv", index=False)

print("\nSaved:")
print("- mxene_flags.csv")
print("- mxene_filtered.csv")


Saved:
- mxene_flags.csv
- mxene_filtered.csv
